# 📋 The Robot & Policy Registry

How does `create_policy("groot")` know what class to load? How does
`sim.add_robot("so100")` find the right MJCF model file?

The answer is the **unified registry** — a JSON-driven system that maps names
to implementations, handles aliases, and hot-reloads when you edit the files.

## Robot Registry

`robots.json` ships with 30+ standard robots. Each entry defines the robot's
simulation model, hardware config, and aliases.

In [ ]:
from strands_robots.registry import (
    resolve_name, get_robot, list_robots,
    has_sim, has_hardware,
)

# Aliases let you be informal — everything resolves to a canonical name
test_names = ["franka", "panda", "SO100_follower", "g1", "cf2", "so101"]
print("Alias resolution:")
for name in test_names:
    print(f"  '{name}' → '{resolve_name(name)}'")

In [ ]:
# Browse the full catalog
robots = list_robots()
print(f"\n{len(robots)} robots available:\n")
for name in robots[:15]:
    info = get_robot(name) or {}
    sim_ok = "🎮" if has_sim(name) else "  "
    hw_ok = "🔧" if has_hardware(name) else "  "
    desc = info.get("description", "")[:48]
    print(f"  {sim_ok}{hw_ok} {name:25s}  {desc}")
if len(robots) > 15:
    print(f"  ... and {len(robots) - 15} more")

In [ ]:
# Inspect a specific robot's full definition
info = get_robot("so100")
if info:
    print("so100 definition:")
    for k, v in info.items():
        print(f"  {k}: {v}")

## Policy Registry

`policies.json` defines how policy providers are found and loaded:

In [ ]:
from strands_robots.registry import (
    list_policy_providers, get_policy_provider, resolve_policy,
)

print("Policy providers:")
for name in list_policy_providers():
    info = get_policy_provider(name) or {}
    desc = info.get("description", "")[:50]
    print(f"  {name:20s}  {desc}")

In [ ]:
# Smart resolution: give it a string, it figures out the provider + kwargs
test_strings = ["mock", "groot", "lerobot/act_aloha_sim", "zmq://localhost:5555"]

print("Smart resolution:")
for s in test_strings:
    try:
        provider, kwargs = resolve_policy(s)
        kw_str = f", kwargs={kwargs}" if kwargs else ""
        print(f"  '{s}' → provider='{provider}'{kw_str}")
    except Exception as e:
        print(f"  '{s}' → {type(e).__name__}: {e}")

## Simulation Model Resolution

When you call `sim.add_robot("so100")`, the model registry finds the MJCF file:

1. **Custom registrations** (`register_urdf()`) — your overrides win
2. **Search paths** — `STRANDS_ASSETS_DIR`, `~/.strands_robots/assets/`, `./assets/`
3. **Menagerie asset manager** — auto-downloads standard robots

In [ ]:
from strands_robots.simulation.model_registry import resolve_model, list_available_models

print("Model resolution:")
for name in ["so100", "panda", "unitree_g1", "crazyflie"]:
    path = resolve_model(name)
    short = "..." + str(path)[-55:] if path else "None"
    print(f"  {name:15s} → {short}")

## Registering Custom Robots

Add your own robots that persist across sessions. They're stored in
`~/.strands_robots/user_robots.json` and overlay on top of the built-in registry.

In [ ]:
from strands_robots.registry import register_robot, list_user_robots, unregister_robot
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    # Create a minimal MJCF file
    with open(os.path.join(tmpdir, "test_arm.xml"), "w") as f:
        f.write("<mujoco><worldbody><body name='link1'/></worldbody></mujoco>")

    # Register it
    register_robot(
        name="test_arm",
        model_xml="test_arm.xml",
        description="My custom test arm",
        category="arm",
        joints=3,
        asset_dir=tmpdir,
        aliases=["testarm"],
    )
    print(f"Registered: {[r['name'] for r in list_user_robots()]}")

    # Clean up
    unregister_robot("test_arm")
    print(f"After cleanup: {list_user_robots()}")

## Hot-Reload

The registry uses file mtime caching. Edit `robots.json` or `policies.json`
and the next query picks up changes — no restart needed.

```python
from strands_robots.registry import reload
reload()  # force re-read all JSON files
```

This makes development fast: add a robot definition, run your sim, see it immediately.